### Importing Packages

In [80]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool, StructuredTool, BaseTool
from pydantic import BaseModel, Field
from typing import Type

### Built-In Tools

In [6]:
seacrh_tool = DuckDuckGoSearchRun()
result = seacrh_tool.invoke("Gold Price Today in Pakistan?")

print(result)

Gold price today per Tola in Pakistan in Pakistani Rupee (PKR) for the most commonly used karats. Also, daily forecasting and updates of gold rates for the most commonly used gold karats in Pakistan; 24K, 22K, 21K, 18K. Historical gold rates and charts. Also check the today gold price in Pakistan of 24k, 22k, 21k, 18k & per Tola, We offer historical & daily gold rates with gold charts & graphs from Karachi gold market. Our gold calculator give you gold prices in different carat ... 21 karat rate for per tola is Rs. 311238 and 18k gold rate is Rs. 266775.00 for 1 tola. Gold Price in Pakistan Today: per oz 948,547.57 Pakistani rupees. Live gold prices for Pakistan today, ensuring that traders, investors, and consumers have access to the most current market data. Detailed information on gold prices per gram for various purities such as 24k, 22k, 21k, 18k, and 14k, as well as gold prices per ounce, tola, and kilogram. Get the latest gold rates in Pakistan. Check today's gold price in Pakis

### Custom Tools - Tool Decorator

In [ ]:
@tool
def addUser(name: str, age: int) -> str:
   """Adds a user with the given name and age."""
   return f"User {name} added with age {age}."

In [14]:
result = addUser.invoke({
    'name': 'Muhammad Abdullah',
    'age': 25
})
print(result)

User Muhammad Abdullah added with age 25.


In [20]:
print(addUser.name)
print(addUser.description)
print(addUser.args)

addUser
Adds a user with the given name and age.
{'name': {'title': 'Name', 'type': 'string'}, 'age': {'title': 'Age', 'type': 'integer'}}


In [21]:
print(addUser.args_schema.model_json_schema())

{'description': 'Adds a user with the given name and age.', 'properties': {'name': {'title': 'Name', 'type': 'string'}, 'age': {'title': 'Age', 'type': 'integer'}}, 'required': ['name', 'age'], 'title': 'addUser', 'type': 'object'}


### Custom Tool - Pydantic and Structured Tool

In [64]:
class UserSchema(BaseModel):
    name: str = Field(required=True, description="Name of the user")
    age: int = Field(required=True, description="Age of the user")
    cnic: str = Field(required=True ,description="CNIC of the user")
    address: str | None = Field(default=None, description="Address of the user")

In [65]:
def addUserWithSchema(name: str, age: int, cnic: str, address: str = None) -> str:
    """Adds a user with the given schema."""
    address_display = address if address else "N/A"
    return f"User {name} added with age {age}, CNIC {cnic}, and address {address_display}."

In [66]:
addUserSchemaTool = StructuredTool.from_function(
    func=addUserWithSchema,
    name="add_user_with_schema",
    description="Adds a user with the given schema.",
    args_schema=UserSchema
)

In [67]:
result = addUserSchemaTool.invoke({
    'name': 'Muhammad Abdullah',
    'age': 25,
    'cnic': '12345-6789012-3',
})

In [68]:
print(addUserSchemaTool.name)
print(addUserSchemaTool.description)
print(addUserSchemaTool.args)

add_user_with_schema
Adds a user with the given schema.
{'name': {'description': 'Name of the user', 'required': True, 'title': 'Name', 'type': 'string'}, 'age': {'description': 'Age of the user', 'required': True, 'title': 'Age', 'type': 'integer'}, 'cnic': {'description': 'CNIC of the user', 'required': True, 'title': 'Cnic', 'type': 'string'}, 'address': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Address of the user', 'title': 'Address'}}


In [31]:
print(addUserSchemaTool.args_schema.model_json_schema())

{'properties': {'name': {'description': 'Name of the user', 'required': True, 'title': 'Name', 'type': 'string'}, 'age': {'description': 'Age of the user', 'required': True, 'title': 'Age', 'type': 'integer'}, 'cnic': {'description': 'CNIC of the user', 'required': True, 'title': 'Cnic', 'type': 'string'}, 'address': {'default': '', 'description': 'Address of the user', 'required': False, 'title': 'Address', 'type': 'string'}}, 'required': ['name', 'age', 'cnic'], 'title': 'UserSchema', 'type': 'object'}


### Custom Tool - BaseTool Class

In [74]:
class UserSchema(BaseModel):
    name: str = Field(required=True, description="Name of the user")
    age: int = Field(required=True, description="Age of the user")
    cnic: str = Field(required=True ,description="CNIC of the user")
    address: str | None = Field(default=None, description="Address of the user")

In [76]:
class CreateUserTool(BaseTool):
    name: str = "create_user"
    description: str = "Creates a user with the given schema."
    args_schema: Type[BaseModel] = UserSchema
    def _run(self, name: str, age: int, cnic: str, address: str = None) -> str:
        """Run the tool with the given user schema."""
        return f"User {name} added with age {age}, CNIC {cnic}, and address {address}."

In [79]:
create_user_tool = CreateUserTool()

create_user_tool.invoke({
    'name': 'Muhammad Abdullah',
    'age': 25,
    'cnic': '12345-6789012-3',
})

'User Muhammad Abdullah added with age 25, CNIC 12345-6789012-3, and address None.'

### ToolKit

In [81]:
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [85]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]


In [87]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers
